In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model
import pickle
import serial
import time
import keyboard  # Instala el paquete 'keyboard' si aún no lo tienes: pip install keyboard

# Establecer conexión serial con Arduino
arduino = serial.Serial('COM3', 115200)  # Cambia 'C|OM6' por el puerto al que está conectado tu Arduino
esp32_2 = serial.Serial('COM7',115200)

# Cargar el modelo entrenado
#modelo_cargado = load_model('modelo_red_neuronal.keras')
modelo_cargado = load_model('modelo_red_neuronal_reducido.keras')
#modelo_cargado = load_model('modelo_red_neuronal_mejor.keras')
#modelo_cargado = load_model('modelo_red_neuronalIVAN.keras')
#modelo_cargado = load_model('modelo_red_neuronal_mejorIVAN.keras')
#modelo_cargado = load_model('modelo_red_neuronalDAVID.keras')
#modelo_cargado = load_model('modelo_red_neuronal_mejorDAVID.keras')
#modelo_cargado = load_model('modelo_red_neuronal_AUMENTADO.keras')
#modelo_cargado = load_model('modelo_red_neuronal_AUMENTADO_mejor.keras')
#modelo_cargado = load_model('modelo_red_neuronal_sinmanocerrada.keras')
#modelo_cargado = load_model('modelo_red_neuronal_sinmanocerrada_mejor.keras')
# Cargar el StandardScaler guardado
with open('standard_scaler.pkl', 'rb') as scaler_file:
    scaler = pickle.load(scaler_file)

# Tiempo de captura en segundos
tiempo_captura = 1  # Capturar datos durante 1 segundo

while True:
    # Tiempo inicial de captura
    tiempo_inicio = time.time()

    # Leer datos desde Arduino durante el tiempo especificado
    datos = []
    while (time.time() - tiempo_inicio) < tiempo_captura:
        linea = arduino.readline().decode().strip()
        if linea:
            try:
                #datos.extend(map(float, linea.split(',')))
                datos.append(float(linea))
            except:
                print("woof")

    # Crear DataFrame con los datos capturados
    df_nuevos_datos = pd.DataFrame({'Dato': datos})

    # Calcular características de los nuevos datos
    N = len(df_nuevos_datos["Dato"])
    MAV = np.mean(np.abs(df_nuevos_datos["Dato"]))
    VAR = np.var(df_nuevos_datos["Dato"])
    RMS = np.sqrt(np.mean(df_nuevos_datos["Dato"] ** 2))
    Wav_Leng = np.sum(np.abs(np.diff(df_nuevos_datos["Dato"])))
    diff_signal = np.diff(df_nuevos_datos["Dato"])
    Dasdv = np.sqrt(np.mean(diff_signal ** 2))
    mean_value = np.mean(df_nuevos_datos["Dato"])
    Damv = np.mean(np.abs(df_nuevos_datos["Dato"] - mean_value))
    iav_value = np.sum(np.abs(df_nuevos_datos["Dato"]))

    # Crear DataFrame con las características calculadas
    df_caracteristicas = pd.DataFrame({
        'MAV_total': [MAV],
        'VAR_total': [VAR],
        'RMS_total': [RMS],
        'Wav_Leng': [Wav_Leng],
        'Dasdv': [Dasdv],
        'Damv': [Damv],
        'iav': [iav_value]
    })

    # Aplicar factores de escalamiento
    factores_escalado = {
        "MAV_total": 1.0,
        "VAR_total": 1.0,
        "RMS_total": 1.0,
        "Wav_Leng": 1.0,
        "Dasdv": 1.0,
        "Damv": 1.0,
        "iav": 1.0
    }

    for caracteristica, factor in factores_escalado.items():
        df_caracteristicas[caracteristica] = df_caracteristicas[caracteristica] * factor

    # Escalar los datos utilizando el mismo escalador que se usó durante el entrenamiento
    X_new = scaler.transform(df_caracteristicas)

    # Realizar predicciones con el modelo cargado
    y_predicciones = modelo_cargado.predict(X_new)

    # Definir las etiquetas de clases
    etiquetas = ['Mano abierta', 'Mano pinza', 'Dedo anular', 'Dedo índice']

    # Obtener la clase predicha (la clase con la probabilidad más alta)
    clase_predicha = np.argmax(y_predicciones, axis=1)[0]

    # Enviar la clase predicha al ESP32
    esp32_2.write(f"{clase_predicha}\n".encode())

    # Mostrar la predicción
    print(f"Predicción (clase): {clase_predicha}, Predicción (etiqueta): {etiquetas[clase_predicha]}")

    # Verificar si se presionó la tecla 'Esc' para salir del bucle
    if keyboard.is_pressed('Esc'):
        print("Saliendo del programa...")
        break

# Cerrar la conexión serial con Arduino al finalizar
arduino.close()


woof
woof
woof
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━

SerialException: WriteFile failed (PermissionError(13, 'El dispositivo no reconoce el comando.', None, 22))